# Category Ontology — Build L1 / L2 / L3 layers

In [1]:
import pandas as pd
from openai import OpenAI
from dotenv import load_dotenv
import os
import json

In [2]:
usda_clean = pd.read_csv("usda_data_dedup_final.csv")

In [3]:
# ── Step 1: Gather all unique categories from the USDA dataset ──────────────

usda_cats = usda_clean["cat"].value_counts().reset_index()
usda_cats.columns = ["original_cat", "count"]
usda_cats["source"] = "usda"

all_cats = usda_cats.copy()
print(f"Total unique categories: {all_cats.original_cat.nunique()}")
print(f"  USDA: {usda_cats.shape[0]}")
all_cats.head()

Total unique categories: 318
  USDA: 318


,original_cat,count,source
0,candy,7789,usda
1,"popcorn, peanuts, seeds & related snacks",5960,usda
2,cookies & biscuits,4400,usda
3,breads & buns,3961,usda
4,"pickles, olives, peppers & relishes",3942,usda


In [4]:
# ── Step 2: Filter to meaningful categories (>=3 items) ──────────────────
# Singletons/tiny cats are noise — we classify them later via item_name fallback
meaningful_cats = all_cats[all_cats["count"] >= 3].copy()
tiny_cats = all_cats[all_cats["count"] < 3]

print(f"Meaningful cats (>=3 items): {len(meaningful_cats)} covering {meaningful_cats['count'].sum():,} rows")
print(f"Tiny cats (<3 items):        {len(tiny_cats)} covering {tiny_cats['count'].sum():,} rows")
print(f"Coverage: {meaningful_cats['count'].sum() / all_cats['count'].sum() * 100:.1f}%")

Meaningful cats (>=3 items): 252 covering 129,967 rows
Tiny cats (<3 items):        66 covering 87 rows
Coverage: 99.9%


In [5]:
meaningful_cats.source.value_counts()

source
usda    252
Name: count, dtype: int64

In [6]:
# ── Step 3: Deduplicate — some cats appear in both sources ───────────────
# Group by original_cat, keep total count and all sources
deduped = (
    meaningful_cats
    .groupby("original_cat")
    .agg(total_count=("count", "sum"), sources=("source", lambda x: ",".join(sorted(set(x)))))
    .reset_index()
    .sort_values("total_count", ascending=False)
)
print(f"Unique categories to classify: {len(deduped)}")
deduped.sources.value_counts()

Unique categories to classify: 252


sources
usda    252
Name: count, dtype: int64

## Phase 2: Define Target Taxonomy & LLM Classification

In [7]:
# ── Step 4: Define the L1 and L2 super-categories ────────────────────────
# L1: ~20 broad groups (based on USDA SR Legacy + a few additions)
# L2: ~60 specific groups (the retrieval filter level)
# L3: the original category string itself (finest grain)

TAXONOMY = {
    "dairy & eggs": [
        "milk", "yogurt", "cheese", "butter & cream",
        "eggs", "ice cream & frozen yogurt", "dairy desserts"
    ],
    "meat": [
        "beef", "pork", "lamb & game", "processed meats & cold cuts",
        "sausages & hot dogs"
    ],
    "poultry": [
        "chicken", "turkey", "other poultry"
    ],
    "fish & seafood": [
        "fresh fish", "shellfish", "canned & smoked fish",
        "frozen fish & seafood"
    ],
    "fruits": [
        "fresh fruits", "dried fruits", "canned & preserved fruits",
        "fruit juices"
    ],
    "vegetables": [
        "fresh vegetables", "frozen vegetables", "canned vegetables",
        "pickles, olives & peppers", "salads"
    ],
    "grains & pasta": [
        "bread", "pasta & noodles", "rice", "flour & corn meal",
        "cereal", "other grains"
    ],
    "baked goods": [
        "cookies & biscuits", "cakes & pastries", "pies & tarts",
        "crackers", "baking mixes"
    ],
    "snacks": [
        "chips & crisps", "popcorn & puffed snacks", "nuts & seeds",
        "snack bars", "other snacks"
    ],
    "sweets & confectionery": [
        "chocolate", "candy & gummy", "chewing gum & mints",
        "honey & syrups", "jams & spreads", "sugar"
    ],
    "beverages": [
        "water", "soda & soft drinks", "tea", "coffee",
        "energy & sport drinks", "alcoholic beverages",
        "powdered drinks", "other beverages"
    ],
    "condiments & sauces": [
        "ketchup & mustard", "salsa & dips", "salad dressing & mayonnaise",
        "cooking sauces", "seasoning & spices", "vinegar & oils"
    ],
    "fats & oils": [
        "vegetable & cooking oils", "butter & margarine", "other fats"
    ],
    "legumes & beans": [
        "canned beans", "dried legumes", "hummus & bean dips"
    ],
    "soups": [
        "canned soup", "prepared soup", "broth & stock"
    ],
    "prepared & frozen meals": [
        "frozen dinners & entrees", "pizza", "sandwiches & wraps",
        "prepared meals", "meal kits"
    ],
    "baby food": [
        "baby food"
    ],
    "supplements": [
        "dietary supplements", "protein powders", "sport nutrition"
    ],
    "plant-based alternatives": [
        "plant-based milk", "plant-based meat", "plant-based other"
    ],
    "other": [
        "other", "undefined"
    ],
}

# Flatten L2 list for the prompt
L2_LIST = []
L1_FOR_L2 = {}
for l1, l2s in TAXONOMY.items():
    for l2 in l2s:
        L2_LIST.append(l2)
        L1_FOR_L2[l2] = l1

print(f"L1 categories: {len(TAXONOMY)}")
print(f"L2 categories: {len(L2_LIST)}")
print(f"\nL2 list:\n{L2_LIST}")

L1 categories: 20
L2 categories: 87

L2 list:
['milk', 'yogurt', 'cheese', 'butter & cream', 'eggs', 'ice cream & frozen yogurt', 'dairy desserts', 'beef', 'pork', 'lamb & game', 'processed meats & cold cuts', 'sausages & hot dogs', 'chicken', 'turkey', 'other poultry', 'fresh fish', 'shellfish', 'canned & smoked fish', 'frozen fish & seafood', 'fresh fruits', 'dried fruits', 'canned & preserved fruits', 'fruit juices', 'fresh vegetables', 'frozen vegetables', 'canned vegetables', 'pickles, olives & peppers', 'salads', 'bread', 'pasta & noodles', 'rice', 'flour & corn meal', 'cereal', 'other grains', 'cookies & biscuits', 'cakes & pastries', 'pies & tarts', 'crackers', 'baking mixes', 'chips & crisps', 'popcorn & puffed snacks', 'nuts & seeds', 'snack bars', 'other snacks', 'chocolate', 'candy & gummy', 'chewing gum & mints', 'honey & syrups', 'jams & spreads', 'sugar', 'water', 'soda & soft drinks', 'tea', 'coffee', 'energy & sport drinks', 'alcoholic beverages', 'powdered drinks', 'o

In [8]:
# ── Step 5: LLM batch classification ─────────────────────────────────────
# We send each unique category → LLM → get back one L2 label
# L1 is derived automatically from the L2→L1 mapping
# L3 is the original category itself

import os
import json
from dotenv import load_dotenv
from openai import OpenAI

load_dotenv()  # loads .env from project root

# ── Choose provider / model ──────────────────────────────────────────────
# Groq (free, fast) — uses the OpenAI-compatible endpoint
client = OpenAI()


# Other Groq models you can try:
#   "llama-3.1-8b-instant"        — fastest, less accurate
#   "mixtral-8x7b-32768"          — good middle ground
#   "gemma2-9b-it"                — Google's 9B

# To switch to OpenAI instead, uncomment these two lines:
# client = OpenAI()               # uses OPENAI_API_KEY from .env
# MODEL = "gpt-4o-mini"           # ~$0.05-0.10 for the full batch
# ─────────────────────────────────────────────────────────────────────────

SYSTEM_PROMPT = f"""You are a food classification expert.
Given a food category name, assign it to exactly ONE of these L2 categories:

{json.dumps(L2_LIST)}

RULES:
- Classify by the PRIMARY food type, not by secondary ingredients.
  "peach pie" → "pies & tarts", NOT "fresh fruits"
  "cheese pizza" → "pizza", NOT "cheese"
- Yogurt drinks, flavored milks → "yogurt" or "milk", NOT "other beverages"
- "Frozen X" → classify by X. "frozen fish" → "frozen fish & seafood"
- Protein powders, supplements → "protein powders" or "dietary supplements"
- Breads, buns, bagels, muffins, flatbreads → "bread"
- If ambiguous, prefer the more specific L2 over "other"
- Reply with ONLY the L2 category name, exactly as listed. Nothing else."""


def classify_category(cat_name: str) -> str:
    resp = client.chat.completions.create(
        model="gpt-4.1-mini",
        messages=[
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": cat_name},
        ],
        temperature=0,
        max_tokens=15,
    )
    return resp.choices[0].message.content.strip().lower()


# Test with a few examples before running the full batch
test_cats = ["breads", "greek-style yogurts", "frozen dinners & entrees",
             "potato crisps", "peanut butters", "protein powders", "dark chocolates"]
for tc in test_cats:
    result = classify_category(tc)
    l1 = L1_FOR_L2.get(result, "UNKNOWN")
    print(f"  {tc:40s} → L2: {result:30s} → L1: {l1}")

  breads                                   → L2: bread                          → L1: grains & pasta
  greek-style yogurts                      → L2: yogurt                         → L1: dairy & eggs
  frozen dinners & entrees                 → L2: frozen dinners & entrees       → L1: prepared & frozen meals
  potato crisps                            → L2: chips & crisps                 → L1: snacks
  peanut butters                           → L2: jams & spreads                 → L1: sweets & confectionery
  protein powders                          → L2: protein powders                → L1: supplements
  dark chocolates                          → L2: chocolate                      → L1: sweets & confectionery


In [9]:
# ── Step 6: Run full batch classification ─────────────────────────────────
# This classifies all meaningful unique categories (~1,600-2,000 strings)
# Cost: ~$0.05-0.10 with gpt-4o-mini

import time

cats_to_classify = deduped["original_cat"].tolist()
print(f"Classifying {len(cats_to_classify)} categories...")

mapping = {}
errors = []

for i, cat in enumerate(cats_to_classify):
    try:
        l2 = classify_category(cat)
        # Validate that the response is in our L2 list
        if l2 not in L2_LIST:
            errors.append((cat, l2, "not in L2 list"))
            l2 = "other"
        mapping[cat] = l2
    except Exception as e:
        errors.append((cat, str(e), "api error"))
        mapping[cat] = "other"

    if (i + 1) % 100 == 0:
        print(f"  {i+1}/{len(cats_to_classify)} done...")

print(f"\nDone! Classified {len(mapping)} categories.")
print(f"Errors/fallbacks: {len(errors)}")
if errors:
    print("Sample errors:")
    for cat, resp, reason in errors[:10]:
        print(f"  {cat} → {resp} ({reason})")

Classifying 252 categories...
  100/252 done...
  200/252 done...

Done! Classified 252 categories.
Errors/fallbacks: 0


In [10]:
# ── Step 7: Save the mapping so we never have to re-classify ──────────────
mapping_df = pd.DataFrame([
    {"original_cat": cat, "cat_l2": l2, "cat_l1": L1_FOR_L2.get(l2, "other")}
    for cat, l2 in mapping.items()
])

mapping_df.to_csv("category_mapping2.csv", index=False)
mapping_df.to_json("category_mapping2.json", orient="records", indent=2)

print(f"Saved {len(mapping_df)} mappings to category_mapping2.csv / .json")
print(f"\nL1 distribution:")
print(mapping_df["cat_l1"].value_counts().to_string())
print(f"\nL2 distribution (top 20):")
print(mapping_df["cat_l2"].value_counts().head(20).to_string())

Saved 252 mappings to category_mapping2.csv / .json

L1 distribution:
cat_l1
prepared & frozen meals     34
other                       23
grains & pasta              22
beverages                   20
baked goods                 19
snacks                      17
vegetables                  17
meat                        17
dairy & eggs                16
condiments & sauces         13
sweets & confectionery      10
fish & seafood               9
supplements                  8
fruits                       7
soups                        6
fats & oils                  4
poultry                      4
legumes & beans              3
plant-based alternatives     2
baby food                    1

L2 distribution (top 20):
cat_l2
other                          21
prepared meals                 18
other snacks                   10
processed meats & cold cuts     9
fresh vegetables                9
frozen dinners & entrees        8
dietary supplements             8
bread                          

## Phase 3: Apply Mapping to Both DataFrames

In [11]:
usda_clean = pd.read_csv("usda_data_dedup_final.csv")

In [12]:
# ── Step 8: Apply the mapping to the USDA dataframe ─────────────────────────
# Load mapping (so this cell works even if you restart the kernel)
cat_map = pd.read_csv("category_mapping2.csv")
l2_lookup = dict(zip(cat_map["original_cat"], cat_map["cat_l2"]))
l1_lookup = dict(zip(cat_map["original_cat"], cat_map["cat_l1"]))

# Map L1 and L2 onto USDA dataframe; L3 = original cat
usda_clean["cat_l1"] = usda_clean["cat"].map(l1_lookup)
usda_clean["cat_l2"] = usda_clean["cat"].map(l2_lookup)
usda_clean["cat_l3"] = usda_clean["cat"]

print("=== USDA coverage ===")
print(f"  L1 mapped: {usda_clean['cat_l1'].notna().sum():,} / {len(usda_clean):,} ({usda_clean['cat_l1'].notna().mean()*100:.1f}%)")
print(f"  L2 mapped: {usda_clean['cat_l2'].notna().sum():,} / {len(usda_clean):,}")
print(f"  L1 unmapped: {usda_clean['cat_l1'].isna().sum():,} rows")

=== USDA coverage ===
  L1 mapped: 129,967 / 130,054 (99.9%)
  L2 mapped: 129,967 / 130,054
  L1 unmapped: 87 rows


In [13]:
# ── Step 9: Handle unmapped rows (tiny categories) ───────────────────────
# For rows where cat wasn't in our mapping (the <3 item categories),
# classify them by item_name using the same LLM function, or assign "other"

unmapped_usda = usda_clean[usda_clean["cat_l2"].isna()]

print(f"Unmapped USDA rows: {len(unmapped_usda)} ({len(unmapped_usda)/len(usda_clean)*100:.1f}%)")

# Option A: assign "other" (simple, fast)
usda_clean["cat_l1"].fillna("other", inplace=True)
usda_clean["cat_l2"].fillna("other", inplace=True)

print("\nAfter filling unmapped → 'other':")
print(f"  USDA L1 unique: {usda_clean['cat_l1'].nunique()}")
print(f"  USDA L2 unique: {usda_clean['cat_l2'].nunique()}")

Unmapped USDA rows: 87 (0.1%)

After filling unmapped → 'other':
  USDA L1 unique: 20
  USDA L2 unique: 78


/var/folders/yz/grwlw1c90b1dbggwlxd8jf200000gn/T/ipykernel_30990/4167241119.py:10: ChainedAssignmentError: A value is being set on a copy of a DataFrame or Series through chained assignment using an inplace method.
Such inplace method never works to update the original DataFrame or Series, because the intermediate object on which we are setting values always behaves as a copy (due to Copy-on-Write).

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' instead, to perform the operation inplace on the original object, or try to avoid an inplace operation using 'df[col] = df[col].method(value)'.

See the documentation for a more detailed explanation: https://pandas.pydata.org/pandas-docs/stable/user_guide/copy_on_write.html
  usda_clean["cat_l1"].fillna("other", inplace=True)
/var/folders/yz/grwlw1c90b1dbggwlxd8jf200000gn/T/ipykernel_30990/4167241119.py:11: ChainedAssignmentError: A value is being set on a copy of a DataFrame or

In [14]:
# ── Step 10: Validate — spot check a few L1 groups ───────────────────────
for l1 in ["dairy & eggs", "fruits", "snacks", "prepared & frozen meals", "beverages"]:
    subset = usda_clean[usda_clean["cat_l1"] == l1]
    sample = subset.sample(min(5, len(subset)), random_state=42)
    print(f"\n{'='*60}")
    print(f"L1: {l1}  ({len(subset):,} rows)")
    print(f"{'='*60}")
    print(sample[["item_name", "cat_l1", "cat_l2", "cat_l3", "source"]].to_string(index=False))


L1: dairy & eggs  (12,274 rows)
                            item_name       cat_l1                    cat_l2                            cat_l3 source
       flavored gels juicy strawberry dairy & eggs            dairy desserts gelatin, gels, pectins & desserts   usda
        bars crushed fruit watermelon dairy & eggs ice cream & frozen yogurt         ice cream & frozen yogurt   usda
coconut greek lowfat pineapple yogurt dairy & eggs                    yogurt                            yogurt   usda
   cream nutritional shake strawberry dairy & eggs                      milk                              milk   usda
       a cream grade pasteurized sour dairy & eggs            butter & cream                             cream   usda

L1: fruits  (592 rows)
                                item_name cat_l1                    cat_l2       cat_l3 source
family fruit gel n orange pack strawberry fruits canned & preserved fruits canned fruit   usda
                           nectar soursop fru

In [18]:
# ── Step 11: Save final dataset ──────────────────────────────────────────
usda_clean.to_csv("usda_dedup_ontology2.csv", index=False)

print(f"Saved usda_dedup_ontology.csv ({len(usda_clean):,} rows)")
print(f"\nColumns: {usda_clean.columns.tolist()}")

Saved usda_dedup_ontology.csv (130,054 rows)

Columns: ['item_name', 'cat', 'carbs_100g', 'kcal_100g', 'fat_100g', 'protein_100g', 'source', 'cat_word_count', 'item_name_word_count', 'cat_l1', 'cat_l2', 'cat_l3']


## Finetuning: edit of the mapping and layers

In [15]:
# Check what landed in "other" — these are likely misclassifications
cat_map = pd.read_csv("category_mapping2.csv")
others = cat_map[cat_map["cat_l2"] == "other"]
print(f"Categories mapped to 'other': {len(others)}")
print(others.sort_values("original_cat").to_string(index=False))

Categories mapped to 'other': 21
                                      original_cat cat_l2 cat_l1
               american indian/alaska native foods  other  other
                                    baked products  other  other
                                baking accessories  other  other
                       baking additives & extracts  other  other
                                    crusts & dough  other  other
                      dough based products / meals  other  other
                       egg rolls, dumplings, sushi  other  other
                                        fast foods  other  other
                                    fish & seafood  other  other
                             liver and organ meats  other  other
meat/poultry/other animals  unprepared/unprocessed  other  other
                                    milk additives  other  other
                                     miscellanious  other  other
                             oral hygiene products  other

In [16]:
# Check a specific L2 to see if anything doesn't belong
cat_map[cat_map["cat_l2"] == "other"].sort_values("original_cat")

,original_cat,cat_l2,cat_l1
183,american indian/alaska native foods,other,other
129,baked products,other,other
187,baking accessories,other,other
82,baking additives & extracts,other,other
77,crusts & dough,other,other
133,dough based products / meals,other,other
232,"egg rolls, dumplings, sushi",other,other
131,fast foods,other,other
59,fish & seafood,other,other
204,liver and organ meats,other,other


In [ ]:
# Search for a specific food and see where it landed
cat_map[cat_map["original_cat"].str.contains("pizza", case=False)]

## EDA of the cats

In [ ]:
usda_ont = pd.read_csv("usda_data_ontology.csv")

In [20]:
usda_ont.columns

Index(['item_name', 'cat', 'carbs_100g', 'kcal_100g', 'fat_100g',
       'protein_100g', 'source', 'cat_word_count', 'item_name_word_count',
       'cat_l1', 'cat_l2', 'cat_l3'],
      dtype='str')

In [21]:
print(usda_ont.cat_l1.unique())

<StringArray>
[     'condiments & sauces',                    'soups',
                    'other',           'grains & pasta',
              'baked goods',                   'snacks',
   'sweets & confectionery',             'dairy & eggs',
                   'fruits',  'prepared & frozen meals',
                     'meat',               'vegetables',
          'legumes & beans',           'fish & seafood',
                'beverages',              'fats & oils',
 'plant-based alternatives',                  'poultry',
              'supplements',                        nan,
                'baby food']
Length: 21, dtype: str


In [22]:
usda_ont[usda_ont.cat_l1 == "baked goods"].head(20)

,item_name,cat,carbs_100g,kcal_100g,fat_100g,protein_100g,source,cat_word_count,item_name_word_count,cat_l1,cat_l2,cat_l3
8,cookies farm ginger pepperidge,biscuits/cookies,78.57,464.0,12.50,3.57,usda,1,4,baked goods,cookies & biscuits,biscuits/cookies
9,chocolate cookies farm graham pepperidge,biscuits/cookies,70.00,467.0,16.67,6.67,usda,1,5,baked goods,cookies & biscuits,biscuits/cookies
11,farm pastry peach pepperidge,sweet bakery products,34.83,281.0,14.61,2.25,usda,3,4,baked goods,cakes & pastries,sweet bakery products
15,cheddar crackers farm pepperidge,biscuits/cookies,66.67,467.0,16.67,10.00,usda,1,4,baked goods,cookies & biscuits,biscuits/cookies
16,cookies farm pepperidge,biscuits/cookies,70.37,481.0,22.22,3.70,usda,1,3,baked goods,cookies & biscuits,biscuits/cookies
18,cherry farm pastry pepperidge,sweet bakery products,30.34,258.0,14.61,2.25,usda,3,4,baked goods,cakes & pastries,sweet bakery products
19,cookies farm pepperidge smores,biscuits/cookies,63.33,500.0,23.33,3.33,usda,1,4,baked goods,cookies & biscuits,biscuits/cookies
21,cookies farm pepperidge strawberry,biscuits/cookies,63.33,467.0,23.33,3.33,usda,1,4,baked goods,cookies & biscuits,biscuits/cookies
22,cookies farm pepperidge peppermint,biscuits/cookies,65.52,517.0,24.14,6.90,usda,1,4,baked goods,cookies & biscuits,biscuits/cookies
29,crackers farm pepperidge,biscuits/cookies,67.86,464.0,16.07,10.71,usda,1,3,baked goods,cookies & biscuits,biscuits/cookies


In [ ]:
usda_ont.cat[usda_ont.cat_l2 == 'other'].unique()
# 5600 rows with 'other' l2 category

<StringArray>
[                    'sauces/spreads/dips/condiments',
                       'dough based products / meals',
 'meat/poultry/other animals  unprepared/unprocessed',
                    'pre packaged fruit & vegetables',
                        'baking additives & extracts',
                    'pizza mixes & other dry dinners',
                                        'other meats',
                                     'crusts & dough',
                                     'fish & seafood',
                                     'milk additives',
                                   'other condiments',
                              'oral hygiene products',
                                 'baking accessories',
                                     'baked products',
                'american indian/alaska native foods',
                                   'restaurant foods',
                                         'fast foods',
                         'soups, sauces, and gravie

In [29]:
usda_ont[(usda_ont.cat_l2 == 'other') & (usda_ont.cat == "other condiments")]


,item_name,cat,carbs_100g,kcal_100g,fat_100g,protein_100g,source,cat_word_count,item_name_word_count,cat_l1,cat_l2,cat_l3
1535,balsamic vinegar white,other condiments,20.00,100.0,0.0,0.0,usda,2,3,other,other,other condiments
1582,balsamic modena of vinegar,other condiments,20.00,100.0,0.0,0.0,usda,2,4,other,other,other condiments
2450,apple butter spread tapn,other condiments,37.50,156.0,0.0,0.0,usda,2,4,other,other,other condiments
2949,balsamic modena of vinegar,other condiments,20.00,113.0,0.0,0.0,usda,2,4,other,other,other condiments
3025,balsamic vinegar,other condiments,20.00,67.0,0.0,0.0,usda,2,2,other,other,other condiments
...,...,...,...,...,...,...,...,...,...,...,...,...
120457,glaze italian,other condiments,60.00,240.0,0.0,0.0,usda,2,2,other,other,other condiments
120497,balsamic premium reduction white,other condiments,93.33,333.0,0.0,0.0,usda,2,4,other,other,other condiments
123337,balsami fig infuse vinegar,other condiments,0.00,267.0,0.0,0.0,usda,2,4,other,other,other condiments
128068,modena of organic vinegar,other condiments,66.67,300.0,0.0,0.0,usda,2,4,other,other,other condiments


## Phase 4: Extend Taxonomy & Fix Unmapped Categories

Manually map the **21 categories** the LLM assigned to `"other"`.  
Strategy:
- **Drop to `other`**: non-food items and overly ambiguous entries (`oral hygiene products`, `miscellanious`, `american indian/alaska native foods`, `baking accessories`)
- **Map to existing L2**: categories that clearly belong to an existing bucket
- **Add new L2s**: categories distinctive enough to warrant their own slot (`crusts & dough`, `baking additives & extracts`, `organ meats`, `other meats`, `egg rolls, dumplings & sushi`, `fast foods & restaurant foods`)

In [31]:
# ── Step 12: Define new L2 additions and manual overrides ────────────────

# New L2 buckets grafted onto the existing taxonomy
TAXONOMY_EXTENSIONS = {
    "baked goods":             ["crusts & dough", "baking additives & extracts"],
    "meat":                    ["organ meats", "other meats"],
    "prepared & frozen meals": ["egg rolls, dumplings & sushi", "fast foods & restaurant foods"],
}

# Extend the L1_FOR_L2 lookup built earlier so patching works correctly
for _l1, _new_l2s in TAXONOMY_EXTENSIONS.items():
    for _l2 in _new_l2s:
        L1_FOR_L2[_l2] = _l1

# Manual mapping for all 21 previously-"other" categories
MANUAL_OVERRIDES = {
    # ── kept as "other" (non-food or too ambiguous) ──────────────────────
    "american indian/alaska native foods":            ("other",                   "other"),
    "oral hygiene products":                          ("other",                   "other"),
    "miscellanious":                                  ("other",                   "other"),
    "baking accessories":                             ("other",                   "other"),

    # ── maps to existing L2 ──────────────────────────────────────────────
    "fish & seafood":                                 ("fish & seafood",          "fresh fish"),
    "other condiments":                               ("condiments & sauces",     "seasoning & spices"),
    "sauces/spreads/dips/condiments":                 ("condiments & sauces",     "cooking sauces"),
    "soups, sauces, and gravies":                     ("soups",                   "prepared soup"),
    "milk additives":                                 ("dairy & eggs",            "milk"),
    "pizza mixes & other dry dinners":                ("baked goods",             "baking mixes"),
    "baked products":                                 ("baked goods",             "baking mixes"),
    "fast foods":                                     ("prepared & frozen meals", "fast foods & restaurant foods"),
    "restaurant foods":                               ("prepared & frozen meals", "fast foods & restaurant foods"),
    "pre packaged fruit & vegetables":                ("vegetables",              "fresh vegetables"),
    "meat/poultry/other animals  unprepared/unprocessed": ("meat",               "other meats"),

    # ── maps to new L2 categories ────────────────────────────────────────
    "crusts & dough":                                 ("baked goods",             "crusts & dough"),
    "dough based products / meals":                   ("baked goods",             "crusts & dough"),
    "baking additives & extracts":                    ("baked goods",             "baking additives & extracts"),
    "egg rolls, dumplings, sushi":                    ("prepared & frozen meals", "egg rolls, dumplings & sushi"),
    "liver and organ meats":                          ("meat",                    "organ meats"),
    "other meats":                                    ("meat",                    "other meats"),
}

new_l2_count = sum(len(v) for v in TAXONOMY_EXTENSIONS.values())
kept_other   = sum(1 for v in MANUAL_OVERRIDES.values() if v[0] == "other")
mapped_new   = sum(1 for v in MANUAL_OVERRIDES.values() if v[1] in L1_FOR_L2 and L1_FOR_L2.get(v[1]) != "other" and v[1] in [l for exts in TAXONOMY_EXTENSIONS.values() for l in exts])
print(f"New L2 categories added to taxonomy : {new_l2_count}")
print(f"Overrides defined                   : {len(MANUAL_OVERRIDES)}")
print(f"  → kept as 'other'                 : {kept_other}")
print(f"  → mapped to existing L2           : {len(MANUAL_OVERRIDES) - kept_other - mapped_new}")
print(f"  → mapped to new L2                : {mapped_new}")

New L2 categories added to taxonomy : 6
Overrides defined                   : 21
  → kept as 'other'                 : 4
  → mapped to existing L2           : 8
  → mapped to new L2                : 9


In [32]:
# ── Step 13: Patch category_mapping2.csv with manual overrides ───────────
import pandas as pd

cat_map = pd.read_csv("category_mapping2.csv")

patched = 0
for orig_cat, (new_l1, new_l2) in MANUAL_OVERRIDES.items():
    mask = cat_map["original_cat"] == orig_cat
    if mask.any():
        cat_map.loc[mask, "cat_l1"] = new_l1
        cat_map.loc[mask, "cat_l2"] = new_l2
        patched += 1

cat_map.to_csv("category_mapping2.csv", index=False)

still_other = cat_map[cat_map["cat_l2"] == "other"]
print(f"Patched {patched} / {len(MANUAL_OVERRIDES)} override rows")
print(f"Remaining 'other' entries after patch: {len(still_other)}")
if len(still_other):
    print(still_other["original_cat"].sort_values().to_string(index=False))

Patched 21 / 21 override rows
Remaining 'other' entries after patch: 4
american indian/alaska native foods
                 baking accessories
                      miscellanious
              oral hygiene products


In [33]:
# ── Step 14: Re-apply updated mapping to USDA dataset & save ─────────────
usda_clean = pd.read_csv("usda_data_dedup_final.csv")

l2_lookup_v2 = dict(zip(cat_map["original_cat"], cat_map["cat_l2"]))
l1_lookup_v2 = dict(zip(cat_map["original_cat"], cat_map["cat_l1"]))

usda_clean["cat_l1"] = usda_clean["cat"].map(l1_lookup_v2).fillna("other")
usda_clean["cat_l2"] = usda_clean["cat"].map(l2_lookup_v2).fillna("other")
usda_clean["cat_l3"] = usda_clean["cat"]

usda_clean.to_csv("usda_data_ontology.csv", index=False)

print(f"Saved usda_data_ontology.csv  ({len(usda_clean):,} rows)")
print(f"\nL1 distribution:")
print(usda_clean["cat_l1"].value_counts().to_string())

Saved usda_data_ontology.csv  (130,054 rows)

L1 distribution:
cat_l1
snacks                      20057
condiments & sauces         15049
sweets & confectionery      14517
baked goods                 13102
dairy & eggs                12719
grains & pasta              11162
vegetables                  10337
beverages                   10004
prepared & frozen meals      8185
meat                         6265
fish & seafood               2975
soups                        2077
fats & oils                   964
plant-based alternatives      810
fruits                        592
legumes & beans               578
poultry                       247
supplements                   228
other                         162
baby food                      24


In [38]:
# ── Step 15: Validate — coverage check & new-L2 spot-check ──────────────
usda_ont = pd.read_csv("usda_dedup_ontology2.csv")

other_rows = usda_ont[usda_ont["cat_l2"] == "other"]
print(f"Rows still in 'other' L2 : {len(other_rows):,}  ({len(other_rows)/len(usda_ont)*100:.1f}%)")
print(f"Unique cats still 'other': {usda_ont.loc[usda_ont['cat_l2'] == 'other', 'cat'].nunique()}")

print("\n── New L2 buckets ──────────────────────────────────────────────────────")
new_l2s = [l2 for exts in TAXONOMY_EXTENSIONS.values() for l2 in exts]
for l2 in new_l2s:
    subset = usda_ont[usda_ont["cat_l2"] == l2]
    print(f"  {l2:40s}  {len(subset):>6,} rows")

print("\n── Sample rows for selected new L2s ───────────────────────────────────")
for l2 in ["crusts & dough", "organ meats", "egg rolls, dumplings & sushi", "fast foods & restaurant foods"]:
    subset = usda_ont[usda_ont["cat_l2"] == l2]
    if len(subset):
        print(f"\n{l2}:")
        print(subset[["item_name", "cat_l1", "cat_l2", "cat"]].head(5).to_string(index=False))

Rows still in 'other' L2 : 5,785  (4.4%)
Unique cats still 'other': 21

── New L2 buckets ──────────────────────────────────────────────────────
  crusts & dough                                 0 rows
  baking additives & extracts                    0 rows
  organ meats                                    0 rows
  other meats                                    0 rows
  egg rolls, dumplings & sushi                   0 rows
  fast foods & restaurant foods                  0 rows

── Sample rows for selected new L2s ───────────────────────────────────


In [54]:
# only choose the rows wihtout "cat_l2" = "other" or "baby food"
usda_final = usda_ont[~usda_ont["cat_l2"].isin(["other", "baby food"])]
usda_final = usda_final[~usda_final["cat_l1"].isin(["other"])]
usda_final = usda_final[~usda_final["cat_l1"].isna()]

In [55]:
usda_final.columns

Index(['item_name', 'cat', 'carbs_100g', 'kcal_100g', 'fat_100g',
       'protein_100g', 'source', 'cat_word_count', 'item_name_word_count',
       'cat_l1', 'cat_l2', 'cat_l3'],
      dtype='str')

In [56]:
usda_final.drop(columns=["cat", "cat_word_count", "item_name_word_count"], inplace=True)

In [58]:
usda_final.to_csv("usda_final.csv", index=False)